In [ ]:
import pandas as pd

In [ ]:
from zoneinfo import ZoneInfo
from datetime import datetime as dt

In [ ]:
dt = dt.datetime.now(ZoneInfo("America/New_York"))

In [10]:
def read_csv(filename: str) -> pd.DataFrame:
    """
    Read DataFrame from a CSV file ``filename`` and convert to a 
    preferred schema.
    """
    df = pd.read_csv(
        filename,
        sep=",",
        na_values="\\N",
        nrows = 100000,
        dtype={
            "station_id": "Int64",
            # Use Pandas Int16 dtype to allow for nullable integers
            "num_bikes_available": "Int16",
            "num_ebikes_available": "Int16",
            "num_bikes_disabled": "Int16",
            "num_docks_available": "Int16",
            "num_docks_disabled": "Int16",
            "is_installed": "Int16",
            "is_renting": "Int16",
            "is_returning": "Int16",
            "station_status_last_reported": "Int64",
            "station_name": str,
            "lat": float,
            "lon": float,
            "region_id": str,
            "capacity": "Int16",
            # Use pandas boolean dtype to allow for nullable booleans
            "has_kiosk": "boolean",
            "station_information_last_updated": "Int64",
            "missing_station_information": "boolean",
        },
    )
    # Read in timestamps as UNIX/POSIX epochs but then convert to the local
    # bike share timezone.
    df["station_status_last_reported"] = pd.to_datetime(
        df["station_status_last_reported"], unit="s", origin="unix", utc=True
    ).dt.tz_convert("America/New_York")

    df["station_information_last_updated"] = pd.to_datetime(
        df["station_information_last_updated"], unit="s", origin="unix", utc=True
    ).dt.tz_convert("America/New_York")
    return df

In [ ]:
df.head()

In [ ]:
df['station_id'].unique()

In [ ]:
correct_ids = set()
for sid in df['station_id'].unique():
    if len(sid) <= 4:
        correct_ids.add(sid)

correct_ids

In [ ]:
df = df[df['station_id'].isin(correct_ids)].copy()

In [ ]:
df[df['station_information_last_reported'] != 'NaT']

In [ ]:
df['num_ebikes_available'].value_counts()

In [ ]:

df = df[(df['station_status_last_reported'] < dt(2018, 12, 31, tzinfo=ZoneInfo('America/New_York'))) & (df['station_status_last_reported'] > dt(2018, 1, 1, tzinfo=ZoneInfo('America/New_York')))].copy()

In [ ]:
df['num_ebikes_available'].isna().sum()

In [ ]:
df.head()

In [ ]:
df = df[['station_id','num_bikes_available','num_docks_available', 'station_status_last_reported']].copy()

In [ ]:
df.head()

In [ ]:
def process_file_original(csv_file):
    def read_csv(filename: str) -> pd.DataFrame:
        """
        Read DataFrame from a CSV file ``filename`` and convert to a 
        preferred schema.
        """
        df = pd.read_csv(
            filename,
            sep=",",
            na_values="\\N",
            dtype={
                "station_id": "Int64",
                # Use Pandas Int16 dtype to allow for nullable integers
                "num_bikes_available": "Int16",
                "num_ebikes_available": "Int16",
                "num_bikes_disabled": "Int16",
                "num_docks_available": "Int16",
                "num_docks_disabled": "Int16",
                "is_installed": "Int16",
                "is_renting": "Int16",
                "is_returning": "Int16",
                "station_status_last_reported": "Int64",
                "station_name": str,
                "lat": float,
                "lon": float,
                "region_id": str,
                "capacity": "Int16",
                # Use pandas boolean dtype to allow for nullable booleans
                "has_kiosk": "boolean",
                "station_information_last_updated": "Int64",
                "missing_station_information": "boolean"
            },
        )
        # Read in timestamps as UNIX/POSIX epochs but then convert to the local
        # bike share timezone.
        df["station_status_last_reported"] = pd.to_datetime(
            df["station_status_last_reported"], unit="s", origin="unix", utc=True
        ).dt.tz_convert("America/New_York")

        df["station_information_last_updated"] = pd.to_datetime(
            df["station_information_last_updated"], unit="s", origin="unix", utc=True
        ).dt.tz_convert("America/New_York")
        return df


    df = read_csv(csv_file)

    correct_ids = set()
    for sid in df['station_id'].unique():
        if len(sid) <= 4:
            correct_ids.add(sid)


    mask = (
        df['station_id'].isin(correct_ids) & 
        (df['station_status_last_reported'] < dt(2018, 12, 31, tzinfo=ZoneInfo('America/New_York'))) & 
        (df['station_status_last_reported'] > dt(2018, 1, 1, tzinfo=ZoneInfo('America/New_York')))
    )


    cols = ['station_id', 'num_bikes_available', 'num_docks_available', 'station_status_last_reported', 'station_name']

    df = df.loc[mask, cols].copy()

    return df

In [ ]:
processed_df = process_file('citi_bike_data_00000.csv')

In [ ]:
STARTTIME = dt(2019, 1, 1, tzinfo=ZoneInfo('America/New_York'))
ENDTIME = dt(2019, 12, 31, tzinfo=ZoneInfo('America/New_York'))
# SPARCITY = 10

def process_file(csv_file):
    def read_csv(filename: str) -> pd.DataFrame:
        """
        Read DataFrame from a CSV file ``filename`` and convert to a 
        preferred schema.
        """
        df = pd.read_csv(
            filename,
            sep=",",
            na_values="\\N",
            # skiprows=lambda x: x % SPARCITY == 0,
            dtype={
                "station_id": str,
                # Use Pandas Int16 dtype to allow for nullable integers
                "num_bikes_available": "Int16",
                "num_ebikes_available": "Int16",
                "num_bikes_disabled": "Int16",
                "num_docks_available": "Int16",
                "num_docks_disabled": "Int16",
                "is_installed": "Int16",
                "is_renting": "Int16",
                "is_returning": "Int16",
                "station_status_last_reported": "Int64",
                "station_name": str,
                "lat": float,
                "lon": float,
                "region_id": str,
                "capacity": "Int16",
                # Use pandas boolean dtype to allow for nullable booleans
                "has_kiosk": "boolean",
                "station_information_last_updated": "Int64",
                "missing_station_information": "boolean"
            },
        )
        # Read in timestamps as UNIX/POSIX epochs but then convert to the local
        # bike share timezone.
        df["station_status_last_reported"] = pd.to_datetime(
            df["station_status_last_reported"], unit="s", origin="unix", utc=True
        ).dt.tz_convert("America/New_York")

        df["station_information_last_updated"] = pd.to_datetime(
            df["station_information_last_updated"], unit="s", origin="unix", utc=True
        ).dt.tz_convert("America/New_York")
        return df


    df = read_csv(csv_file)

    correct_ids = set()
    for sid in df['station_id'].unique():
        if len(sid) <= 4:
            correct_ids.add(sid)


    mask = (
        df['station_id'].isin(correct_ids) & 
        (df['station_status_last_reported'] < ENDTIME) & 
        (df['station_status_last_reported'] > STARTTIME)
    )


    cols = ['station_id', 'num_bikes_available', 'num_docks_available', 'station_status_last_reported']

    df = df.loc[mask, cols].copy()

    return df


In [ ]:
processed_df = process_file_original('citi_bike_data_00000.csv')

In [ ]:
processed_df = process_file('citi_bike_data_00000.csv')

In [ ]:
len(df)

In [ ]:
df = read_csv('citi_bike_data_00001.csv')

In [ ]:
correct_ids = set(df['station_id'].unique())

print(correct_ids)
print(len(correct_ids))

id_to_name = {}
counter = 0

for row in df.itertuples():
    if not pd.isna(row.station_name):
        if row.station_id in correct_ids:
            sid = row.station_id
            name = row.station_name
            id_to_name[sid] = name
            correct_ids.discard(sid)
            print("Found", sid)
    if len(correct_ids) == 0:
        break
    counter += 1

print("Remaining IDs:", correct_ids)
print("ID to Name mapping:", id_to_name)

In [ ]:
pd.isna(df.loc[3800, 'station_name'])

In [ ]:
df[df['station_id'] == '430']['station_name'].unique()

In [ ]:
df[df['station_name'] == 'Centre St & Worth St']

In [ ]:
processed_df['station_id'].unique()

In [ ]:
df['station_id'].unique()

In [ ]:
df['station_id'].unique()

In [ ]:
import pandas as pd

stations_df = pd.read_csv('citibike-stations.csv')

id_to_shortname = dict(zip(stations_df['station_id'], stations_df['short_name']))

In [ ]:
stations_df.dtypes

In [ ]:
stations_df[stations_df['station_id'] == 3934]

In [11]:
df = read_csv('citi_bike_data_00001.csv')

In [22]:
df.tail(80)

,station_id,num_bikes_available,num_ebikes_available,num_bikes_disabled,num_docks_available,num_docks_disabled,is_installed,is_renting,is_returning,station_status_last_reported,station_name,lat,lon,region_id,capacity,has_kiosk,station_information_last_updated,missing_station_information
99920,3853,2,0,0,18,0,1,1,1,2019-12-14 15:53:33-05:00,NaN,NaN,NaN,NaN,<NA>,<NA>,NaT,True
99921,3808,0,0,0,19,0,1,1,1,2020-07-06 16:28:32-04:00,NaN,NaN,NaN,NaN,<NA>,<NA>,NaT,True
99922,386,31,0,2,49,0,1,1,1,2020-08-06 07:52:52-04:00,Centre St & Worth St,40.714948,-74.002345,71,43,True,2019-08-01 16:47:30-04:00,False
99923,3808,0,0,1,18,0,1,1,1,2020-10-29 01:46:21-04:00,NaN,NaN,NaN,NaN,<NA>,<NA>,NaT,True
99924,386,78,3,3,1,0,1,1,1,2020-09-21 14:07:38-04:00,Centre St & Worth St,40.714948,-74.002345,71,43,True,2019-08-01 16:47:30-04:00,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,386,32,0,3,8,0,1,1,1,2020-05-20 13:50:00-04:00,Centre St & Worth St,40.714948,-74.002345,71,43,True,2019-08-01 16:47:30-04:00,False
99996,223,21,0,2,10,0,1,1,1,2021-12-04 15:26:10-05:00,W 13 St & 7 Ave,40.737815,-73.999947,71,33,True,2019-08-01 16:47:30-04:00,False
99997,223,19,0,0,14,0,1,1,1,2020-07-06 20:44:44-04:00,W 13 St & 7 Ave,40.737815,-73.999947,71,33,True,2019-08-01 16:47:30-04:00,False
99998,386,26,0,1,16,0,1,1,1,2019-10-11 12:49:53-04:00,Centre St & Worth St,40.714948,-74.002345,71,43,True,2019-08-01 16:47:30-04:00,False


In [25]:
df[df['station_id'] ==  386]['station_name'].unique()

<StringArray>
[nan, 'Centre St & Worth St']
Length: 2, dtype: str